<a href="https://colab.research.google.com/github/Doop318/business-predictive-analytics/blob/main/Data_Cleaning_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
### **Attribution & Reference**
This project is a replication and pedagogical extension of the original data preprocessing work created by [Roy Jafari](https://github.com/royjafari/).

* **Original Repository:** [data-cleaning-example-2](https://github.com/data-cleaning-example-2)
* **Companion Video Tutorial:** [Data Cleaning Example - Unwise Data Collection](https://www.youtube.com/watch?v=aVklIqpI-ZI&list=PLeIx1CWnEc7tji28Xb2j0Qq6gvu_AYQgq&index=35)

_Note: This notebook has been re-structured with beginner-friendly explanations and step-by-step interpretation guides to assist learners in mastering basic raw data restructuring and text database normalization techniques._

---

# **Background Information & The Problem We Are Solving**

Imagine you are trying to study the speeches given by a political figure across various campaign rallies. Instead of having all the information neatly organized in a single spreadsheet, the data has been collected in an "unwise" or raw structure:

1. **The Speech Files:** You have a folder called [`Speeches`](https://github.com/royjafari/data-cleaning-example-2/tree/main/Speeches) containing 35 separate text files. Each file contains the full text transcript of a single speech.
2. **Hidden Information in File Names:** The text files themselves do not have the location or date written inside them. Instead, that crucial information is hidden right in the name of each file (for example: [`BattleCreekDec19_2019.txt`](https://github.com/royjafari/data-cleaning-example-2/blob/main/Speeches/BattleCreekDec19_2019.txt) or [`PhoenixFeb19_2020.txt`](https://github.com/royjafari/data-cleaning-example-2/blob/main/Speeches/PhoenixFeb19_2020.txt)).

## **The Big Question & Task**
We want to extract the hidden details from the file names, count the lengths of the speeches, and bring all 35 text documents together into one structured table. Our goal is to create a clean pandas DataFrame with exactly these four columns:
* **City:** Where the rally took place.
* **Date:** The exact calendar day it happened (stored as a real date object, not text).
* **Speech:** The actual words spoken.
* **NumberOfWords:** A count of how long the speech was.

---

## **Why Do We Need to Conduct Data Cleaning?**

In text-based data science (Natural Language Processing), data rarely arrives in a tidy format. We must perform data cleaning for several vital reasons:

1. **Re-structuring Text Data for Analytics:** Computers cannot run analytics, sort timelines, or build statistical models across 35 separate unorganized text files. This level of data cleaning isn't about altering the words spoken; it is about structural restructuring—extracting metadata from text strings so that the values become measurable and manipulatable.
2. **Standardizing Strings into Machine-Readable Types:** A string like `Dec19_2019` is just meaningless text characters to a computer. It doesn't know that Dec comes after Nov. By cleaning and translating these text components into a standardized datetime object, we unlock the ability to chronologically sort, timeline-filter, and analyze changes over time.

---

## **Step-by-Step Data Cleaning Guide**

### **Step 1: Scan the Directory and Gather File Names**
First, we tell Python to look inside the `Speeches` folder and make a list of every file name it finds.

---

In [7]:
import os
import pandas as pd
from os import listdir

# Clone the repository to get the 'Speeches' directory locally
if not os.path.exists('data-cleaning-example-2'):
    !git clone https://github.com/royjafari/data-cleaning-example-2.git

# Read the list of file names from the cloned directory
FileNames = listdir('data-cleaning-example-2/Speeches')
print(f"Total speech files catalogued: {len(FileNames)}")

Total speech files catalogued: 35


### **Step 2: Initialize an Empty DataFrame Workspace**
We build an empty table with 35 blank rows (one for each file) and prepare our base columns.

---

In [8]:
speech_df = pd.DataFrame(index = range(len(FileNames)), columns = ['FileName', 'Speech'])

speech_df.head(3)

,FileName,Speech
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN


### **Step 3: Loop Through and Ingest the Text Files**
We write a loop that opens each file one by one, reads the text content, and places both the name of the file and the speech text into our table.

---

### **Enumerate() in Python**
https://www.geeksforgeeks.org/python/enumerate-in-python/

In [9]:
for i, f_name in enumerate(FileNames):
  f = open('data-cleaning-example-2/Speeches/' + f_name, 'r', encoding = 'utf-8')
  f_content = f.readlines()
  f.close()
  speech_df.at[i, 'FileName'] = f_name
  speech_df.at[i, 'Speech'] = f_content[0]

speech_df.head(3)

,FileName,Speech
0,MilwaukeeJan14_2020.txt,Well thank you very much. And I'm thrilled to ...
1,WildwoodJan28_2020.txt,Thank you. Thank you. I love New Jersey and I'...
2,LasVegasFeb21_2020.txt,"Well, thank you very much. And hello Las Vegas..."


### **Step 4: Extract the City Name Using String Splitting**
Since the city name always comes right before the month abbreviations **(e.g, `MilwaukeeJan14_2020`.txt)**, we use a predefined list of month prefixes to find where the city name ends, pulling out just the city characters.


---

In [22]:
Months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Oct','Sep','Nov','Dec']

def separateCity(v):
  for month in Months:
    if (month in v):
      return v[:v.find(month)]

In [23]:
speech_df['City'] = speech_df.FileName.apply(separateCity)

speech_df.head(3)

,FileName,Speech,City
0,MilwaukeeJan14_2020.txt,Well thank you very much. And I'm thrilled to ...,Milwaukee
1,WildwoodJan28_2020.txt,Thank you. Thank you. I love New Jersey and I'...,Wildwood
2,LasVegasFeb21_2020.txt,"Well, thank you very much. And hello Las Vegas...",LasVegas



### **Step 5: Extract and Format the Raw Date String**
Now we isolate the middle section of the file name (after the city name and before the `.txt` suffix) to capture our raw month, day, and year characters.


---


In [24]:
def separateDate(r):
  Date = r.FileName[len(r.City):r.FileName.find('.txt')]
  return Date[:3] + '_' + Date[3:]

In [25]:
speech_df['Date'] = speech_df.apply(separateDate, axis = 1)
speech_df.head(3)

,FileName,Speech,City,Date
0,MilwaukeeJan14_2020.txt,Well thank you very much. And I'm thrilled to ...,Milwaukee,Jan_14_2020
1,WildwoodJan28_2020.txt,Thank you. Thank you. I love New Jersey and I'...,Wildwood,Jan_28_2020
2,LasVegasFeb21_2020.txt,"Well, thank you very much. And hello Las Vegas...",LasVegas,Feb_21_2020



### **Step 6: Convert Text Months to Calendar Numbers**
To make a true date format, text like "Dec" needs to become numerical "12". We expand our date string into separate columns and swap text months with real numbers.

---


In [26]:
date_df = speech_df.Date.str.split('_', expand = True)
date_df.columns = ['Month', 'Day', 'Year']
date_df.head(3)

,Month,Day,Year
0,Jan,14,2020
1,Jan,28,2020
2,Feb,21,2020


In [27]:
# Map each three-letter text string to its calendar integer equivalent
replace_dic = {Months[i]: i+1 for i in range(12)}
date_df['Month'] = date_df['Month'].replace(replace_dic)
date_df.head(3)

/tmp/ipykernel_3398/2854331989.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  date_df['Month'] = date_df['Month'].replace(replace_dic)


,Month,Day,Year
0,1,14,2020
1,1,28,2020
2,2,21,2020



### **Step 7: Finalize True Datetime Conversion and Chronological Sort**
We use Python's built-in datetime tool to convert those numerical fields into real datetime objects, assign them back to our table, and sort our history cleanly from oldest to newest.


---


In [29]:
import datetime

def TODateTime(r):
  return datetime.date(int(r.Year), r.Month, int(r.Day))

In [31]:
speech_df.Date = date_df.apply(TODateTime, axis = 1)
speech_df.sort_values('Date')
speech_df.head(3)

,FileName,Speech,City,Date
0,MilwaukeeJan14_2020.txt,Well thank you very much. And I'm thrilled to ...,Milwaukee,2020-01-14
1,WildwoodJan28_2020.txt,Thank you. Thank you. I love New Jersey and I'...,Wildwood,2020-01-28
2,LasVegasFeb21_2020.txt,"Well, thank you very much. And hello Las Vegas...",LasVegas,2020-02-21



### **Step 8: Count the Words in Each Speech**
We calculate the length of the speeches by breaking the text string apart wherever spaces occur and evaluating the total size of the resulting word array.



---


In [32]:
def countWords(r):
  return len(r.Speech.split(' '))


In [33]:
speech_df['NumberOfWords'] = speech_df.Speech.apply(lambda v:len(v.split(' ')))
speech_df.head(3)

,FileName,Speech,City,Date,NumberOfWords
0,MilwaukeeJan14_2020.txt,Well thank you very much. And I'm thrilled to ...,Milwaukee,2020-01-14,9467
1,WildwoodJan28_2020.txt,Thank you. Thank you. I love New Jersey and I'...,Wildwood,2020-01-28,6966
2,LasVegasFeb21_2020.txt,"Well, thank you very much. And hello Las Vegas...",LasVegas,2020-02-21,13809


In [34]:
speech_df = speech_df[['City', 'Date', 'Speech', 'NumberOfWords']]
speech_df.head(3)

,City,Date,Speech,NumberOfWords
0,Milwaukee,2020-01-14,Well thank you very much. And I'm thrilled to ...,9467
1,Wildwood,2020-01-28,Thank you. Thank you. I love New Jersey and I'...,6966
2,LasVegas,2020-02-21,"Well, thank you very much. And hello Las Vegas...",13809



### **Step 9: Export the Integrated Master Files**
Now that the data is integrated and clean, we export our data into a clean CSV file ready for analytics software.



In [35]:
speech_df.to_csv('InegratedSpeech.csv', index = False)

Open the CSV file with a notepad and Excel and compare them with Pandas DataFrame format.

In [36]:
speech_df.to_csv('InegratedSpeech.json')